# Agentic AI Test Case Generator
### Capstone Project — Google Colab + Groq

**Workflow:** Requirement → Generate → Critique → Improve → Structure → Validate → Report → Export

This notebook generates requirement-traceable test suites for:
- Feature A — User Login
- Feature B — Apply Promo Code at Checkout

The supplied business requirements and acceptance criteria are preserved. The notebook adds deterministic QA gates,
secure credential handling, consistent model configuration, coverage reporting, a design writeup, and an evaluator-oriented reflection.

**Credential rule:** Store `GROQ_API_KEY` in Google Colab Secrets. Never paste the key into a code cell or commit it to GitHub.


## 1. Install libraries
Run this cell first.

In [1]:
# Google Colab dependency installation
!pip install -q "groq>=0.31.0" pandas openpyxl json-repair reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.7 MB/s eta 0:00:00


## 2. Imports and Groq connection
In Colab, click the **Secrets (🔑)** icon → add `GROQ_API_KEY` → enable notebook access.

In [2]:
# Colab-safe imports
import os
import json
import re
import time
import pandas as pd

try:
    from google.colab import userdata
except ImportError:
    userdata = None

from groq import Groq


In [3]:
# Secure Groq authentication and stable model configuration
api_key = None
if userdata is not None:
    try:
        api_key = userdata.get("GROQ_API_KEY")
    except Exception:
        api_key = None

api_key = api_key or os.environ.get("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY is required. In Google Colab, open the Secrets (key) panel, "
        "create GROQ_API_KEY, and enable notebook access. Never hardcode the key."
    )

MODEL = "openai/gpt-oss-120b"
client = Groq(api_key=api_key)
available_models = [m.id for m in client.models.list().data]
if MODEL not in available_models:
    raise RuntimeError(
        f"Configured model '{MODEL}' is not available for this API key.\n"
        "Available models include: " + ", ".join(available_models[:30])
    )
print(f"Groq authentication successful. Model: {MODEL}")


Groq authentication successful. Model: openai/gpt-oss-120b


In [4]:
# Optional model availability check
try:
    available_models = [m.id for m in client.models.list().data]
    print("Configured model available:", MODEL in available_models)
    if MODEL not in available_models:
        raise RuntimeError(
            f"Configured model '{MODEL}' is not available for this API key. "
            "Choose an available text-generation model and update MODEL."
        )
except Exception as exc:
    raise RuntimeError(f"Groq model availability check failed: {exc}") from exc


Configured model available: True


## 3. Test the API
If this cell returns an answer, the connection is working.

In [5]:
# Lightweight API smoke test
try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a software testing expert."},
            {"role": "user", "content": "Explain positive testing in one sentence."}
        ],
        temperature=0,
        max_completion_tokens=120
    )
    print(response.choices[0].message.content.strip())
except Exception as exc:
    raise RuntimeError(f"Groq API smoke test failed: {exc}") from exc


Positive testing verifies that a system behaves as expected when given valid, typical inputs and conditions, confirming that it correctly performs its intended functions.


## 4. Reusable JSON call with retry
The agent uses structured JSON responses so the generated test suite can be exported to CSV/Excel/Gherkin.

In [6]:
def _bounded_completion_tokens(prompt, requested, hard_cap=2200):
    """Keep each request safely below an 8K-token request budget."""
    estimated_input_tokens = max(1, len(prompt) // 4)
    safety_budget = 7000
    allowed = max(256, safety_budget - estimated_input_tokens)
    return min(requested, hard_cap, allowed)


def call_json_agent(system_prompt, user_prompt, max_tokens=1800, retries=3):
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            completion_tokens = _bounded_completion_tokens(user_prompt, max_tokens)
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                temperature=0,
                max_completion_tokens=completion_tokens,
            )
            raw = (response.choices[0].message.content or "").strip()
            if not raw:
                raise ValueError("Model returned an empty response.")
            raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.I)
            raw = re.sub(r"\s*```$", "", raw).strip()
            return json.loads(raw)
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(2 * attempt)
    raise RuntimeError(f"JSON agent failed after {retries} attempts: {last_error}") from last_error


def ask_text_agent(system_prompt, user_prompt, max_tokens=1600):
    completion_tokens = _bounded_completion_tokens(user_prompt, max_tokens)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        temperature=0,
        max_completion_tokens=completion_tokens
    )
    return (response.choices[0].message.content or "").strip()


## 5. Feature specifications

In [7]:
feature_a = """
Feature: User Login

User Story:
As a registered user, I want to log in with my email and password
so that I can access my account.

Description:
The login screen has an Email field, a Password field, a Log In button,
and a Forgot password? link. On successful login the user is taken to
their dashboard. On failure an error message is shown.

Acceptance Criteria:

AC1 — Valid login.
Given a registered, active user, when they enter their correct email
and password and click Log In, then they are redirected to the dashboard.

AC2 — Invalid password.
Given a registered user, when they enter a correct email but wrong
password, then an error "Invalid email or password" is shown and they
remain on the login page.

AC3 — Unregistered email.
When an email that is not registered is entered with any password,
then the same generic "Invalid email or password" error is shown
(no indication of whether the email exists).

AC4 — Empty fields.
When either field is left blank and Log In is clicked, then inline
validation prompts the user to fill the required field(s); no request
is sent.

AC5 — Email format.
When the email field contains a value that is not a valid email format,
then an inline "Enter a valid email address" message is shown.

AC6 — Account lockout.
After 5 consecutive failed attempts within 15 minutes, the account is
temporarily locked for 30 minutes and a "Your account is locked.
Try again later." message is shown, even if correct credentials are
subsequently entered.

AC7 — Case sensitivity.
The email is case-insensitive (User@x.com == user@x.com);
the password is case-sensitive.

AC8 — Session.
On successful login a session is established; on browser refresh the
user remains logged in until the session expires (24 hours) or they log out.

AC9 — Inactive account.
A user whose account is deactivated sees "This account is inactive.
Contact support." and is not logged in.

Notes:
Passwords are between 8 and 64 characters.
The Forgot password? flow is out of scope.
"""

In [8]:
feature_b = """
Feature: Apply Promo Code at Checkout

User Story:
As a shopper, I want to apply a promo code at checkout so that I
receive a discount on my order.

Description:
At checkout there is a Promo code input and an Apply button.
When a valid code is applied, the discount is reflected in the
order summary and the total updates. An invalid or ineligible code
shows an error and the total is unchanged.

Promo code rules:

- Percentage codes, e.g. SAVE10 = 10% off the item subtotal.
- Fixed amount codes, e.g. FLAT200 = ₹200 off the item subtotal.
- Codes are case-insensitive.
- Some codes require a minimum subtotal.
- Expired codes are rejected.
- A single-use code cannot be reused by the same customer.
- Only one code may be applied per order.
- Applying a second code replaces the first only after confirmation.
- Fixed discount cannot make subtotal negative.
- Shipping and taxes are calculated on the discounted subtotal.

Acceptance Criteria:

AC1 — Valid percentage code.
Applying SAVE10 to a ₹1000 subtotal reduces it by ₹100;
the order total updates accordingly.

AC2 — Valid fixed code above minimum.
Applying FLAT200 to a ₹1500 subtotal reduces it by ₹200.

AC3 — Fixed code below minimum.
Applying FLAT200 to an ₹800 subtotal is rejected with
"This code requires a minimum order of ₹1000."

AC4 — Expired code.
Applying an expired code shows "This code has expired."
and the total is unchanged.

AC5 — Invalid code.
Applying a non-existent code shows "Invalid promo code."
and the total is unchanged.

AC6 — Case insensitivity.
save10 behaves identically to SAVE10.

AC7 — Already used.
Reapplying a single-use code already redeemed by the customer
shows "This code has already been used."

AC8 — Discount cap.
Applying FLAT200 to a ₹150 subtotal results in a subtotal of ₹0,
never negative.

AC9 — Replace existing code.
Applying a second code prompts the user to replace the first;
on confirm, only the new discount applies.

AC10 — Empty input.
Clicking Apply with no code shows "Enter a promo code."

AC11 — Whitespace/format.
Leading/trailing spaces are trimmed before validation.

AC12 — Recalculation on cart change.
If the cart changes after a code is applied and the subtotal drops
below the minimum, the discount is re-validated and removed if no
longer eligible.
"""

In [9]:
def _estimate_tokens(text):
    """Conservative rough token estimate for rate-limit budgeting."""
    return max(1, len(str(text)) // 4)


# Groq free/on-demand accounts can enforce a rolling tokens-per-minute limit.
# Keep a local rolling budget so Feature A and Feature B do not collide.
REQUEST_TPM_BUDGET = 7000
_recent_token_requests = []


def _wait_for_token_budget(estimated_tokens):
    global _recent_token_requests
    while True:
        now = time.monotonic()
        _recent_token_requests = [
            (ts, tokens) for ts, tokens in _recent_token_requests
            if now - ts < 60
        ]
        used = sum(tokens for _, tokens in _recent_token_requests)

        if used + estimated_tokens <= REQUEST_TPM_BUDGET:
            _recent_token_requests.append((now, estimated_tokens))
            return

        oldest_ts = min(ts for ts, _ in _recent_token_requests)
        sleep_for = max(1.0, 60 - (now - oldest_ts) + 0.5)
        print(f"⏳ Groq TPM pacing: waiting {sleep_for:.1f}s before the next AI call...")
        time.sleep(sleep_for)


def _bounded_completion_tokens(prompt, requested, hard_cap=1400):
    estimated_input_tokens = _estimate_tokens(prompt)
    # Keep input + requested completion comfortably below the 8K limit.
    safety_budget = 6000
    allowed = max(256, safety_budget - estimated_input_tokens)
    return min(requested, hard_cap, allowed)


def ask_ai(prompt, temperature=0, max_tokens=1400, retries=4):
    """Colab-safe Groq call with prompt bounds, TPM pacing, and retry handling."""
    MAX_PROMPT_CHARS = 16000
    if len(prompt) > MAX_PROMPT_CHARS:
        prompt = prompt[:MAX_PROMPT_CHARS] + "\n[Prompt shortened to stay within the API budget.]"

    last_error = None

    for attempt in range(1, retries + 1):
        completion_tokens = _bounded_completion_tokens(
            prompt, max_tokens, hard_cap=1400
        )
        estimated_request_tokens = _estimate_tokens(prompt) + completion_tokens
        _wait_for_token_budget(estimated_request_tokens)

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a senior software QA engineer. "
                            "Create accurate, requirement-traceable software test cases. "
                            "Never invent requirements or functionality."
                        )
                    },
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                max_completion_tokens=completion_tokens
            )

            text = (response.choices[0].message.content or "").strip()
            if not text:
                raise ValueError("AI returned an empty response.")
            return text

        except Exception as exc:
            last_error = exc
            error_text = str(exc).lower()

            # Retry transient Groq rate-limit responses.
            if ("rate_limit" in error_text or "rate limit" in error_text
                    or "429" in error_text or "413" in error_text):
                if attempt < retries:
                    wait_seconds = min(30, 5 * attempt)
                    print(
                        f"⚠️ Groq rate limit encountered. "
                        f"Retrying in {wait_seconds}s (attempt {attempt}/{retries})..."
                    )
                    time.sleep(wait_seconds)
                    continue

            raise

    raise RuntimeError(f"Groq AI call failed after {retries} attempts: {last_error}") from last_error


In [10]:
def generate_test_cases(requirement):

    prompt = f"""
You are a senior QA test case designer.

Read the following requirement carefully:

{requirement}

Generate a comprehensive draft test suite.

Each test case must include:

- Test Case ID
- Acceptance Criteria ID
- Test Scenario
- Preconditions
- Test Data
- Test Steps
- Expected Result
- Category
- Priority
- Risk

Categories:

Positive
Negative
Boundary
Edge

Rules:

1. Every acceptance criterion must be considered.
2. Test cases must be traceable to an AC.
3. Do not invent functionality.
4. Include boundary conditions from the requirement.
5. Include important timing and threshold conditions.
6. Include negative scenarios.
7. Include edge cases.
8. Mark high-risk scenarios appropriately.
"""

    return ask_ai(prompt, max_tokens=1400)

In [11]:
draft_a = generate_test_cases(feature_a)

print(draft_a)

**Draft Test Suite – User Login Feature**  
*(All test cases are traceable to the Acceptance Criteria (AC) listed in the requirement.  No functionality beyond the specification has been added.)*  

| # | Test Case ID | Acceptance Criteria ID | Test Scenario | Preconditions | Test Data | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|------------------------|---------------|---------------|-----------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Successful login with valid credentials | User **active** and **registered** in the system. | Email: `user@example.com`  (registered) <br>Password: `ValidPass123!` (8‑64 chars) | 1. Open login page.<br>2. Enter email.<br>3. Enter password.<br>4. Click **Log In**. | User is redirected to the Dashboard. A session cookie is created. | Positive | High | High (core functionality) |
| 2 | TC‑002 | AC2 | Invalid password – generic error shown | User **active** and **registered**. | E

In [12]:
draft_b = generate_test_cases(feature_b)

print(draft_b)

**Draft Test Suite – “Apply Promo Code at Checkout”**  
*(All test cases are traceable to the Acceptance Criteria (AC) listed in the requirement. No functionality beyond the specification has been introduced.)*  

| # | Test Case ID | Acceptance Criteria ID | Test Scenario | Preconditions | Test Data | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|------------------------|---------------|---------------|-----------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Apply a valid percentage promo code (SAVE10) to an order whose subtotal meets the rule. | • User is logged in (or guest) <br>• Cart contains items totalling **₹1,000** (item subtotal) <br>• No promo code currently applied | Promo code: **SAVE10** (case‑insensitive) | 1. Navigate to Checkout page.<br>2. Verify subtotal = ₹1,000.<br>3. Enter **SAVE10** in the Promo‑code field.<br>4. Click **Apply**. | Discount of 10 % (₹100) is applied.<br>• New item subtotal = ₹

## 7. Critic Agent
The critic checks the draft against every acceptance criterion and identifies gaps.

In [13]:
def critique_test_cases(feature, draft):
    prompt = f"""
You are an expert software QA reviewer.

FEATURE / REQUIREMENT:
{feature}

DRAFT TEST CASES:
{draft}

Review the draft without rewriting the final suite.
For EVERY acceptance criterion, determine coverage and identify missing or weak scenarios.
Identify duplicates, unsupported assumptions, and traceability issues.
Also check Positive, Negative, Boundary, Edge, Priority, Risk, thresholds, timing,
state changes, error messages, session behavior, repeated attempts, and cart recalculation where applicable.

Return a concise QA critique containing:
COVERAGE REPORT
DUPLICATES
UNSUPPORTED ASSUMPTIONS
RISK GAPS
RECOMMENDATIONS

Do not invent new business requirements.
"""
    return ask_ai(prompt, max_tokens=1200)


The critic agent is defined once in Cell 19 and reused for both features.


In [14]:
critique_a = critique_test_cases(
    feature_a,
    draft_a
)

print(critique_a)

**QA Critique – User Login Feature (draft suite)**  

---

## 1. COVERAGE REPORT (by Acceptance Criterion)

| AC | Covered by Draft TC(s) | Gaps / Weaknesses |
|----|------------------------|-------------------|
| **AC1 – Valid login** | TC‑001 (positive) | – No verification of *session cookie* creation, *dashboard UI* elements, or *post‑login redirect* timing. |
| **AC2 – Invalid password** | TC‑002 (negative) | – Does not confirm that the error message is **exactly** “Invalid email or password”. No check that no session is created. |
| **AC3 – Unregistered email** | TC‑003 (negative) | – Same as AC2 – only checks message text, not that the system does **not** disclose existence of the email. |
| **AC4 – Empty fields** | TC‑004, TC‑005, TC‑006 (negative) | – Only verifies UI inline text; does **not** verify that *no network request* is sent (needs dev‑tools capture). No test for *whitespace‑only* input. |
| **AC5 – Email format** | TC‑007, TC‑008 (negative) | – Covers two malformed pa

## Critic definition consolidated


In [15]:
critique_b = critique_test_cases(
    feature_b,
    draft_b
)

print(critique_b)

⏳ Groq TPM pacing: waiting 51.1s before the next AI call...
⏳ Groq TPM pacing: waiting 3.3s before the next AI call...
**QA Critique – “Apply Promo Code at Checkout” Draft Suite**  

---

## 1. Coverage Report (per Acceptance Criterion)

| AC | Covered by Test(s) | Gaps / Weaknesses |
|----|--------------------|-------------------|
| **AC1 – Valid % code** | TC‑001 (positive) | – No check of rounding (e.g., ₹999 → 10 % = ₹99.90). <br>– No verification that discount appears as a separate line item and that only the *item subtotal* is reduced (shipping/tax unchanged). |
| **AC2 – Valid fixed code above minimum** | TC‑003 (positive) | – No boundary test where subtotal = exact minimum (e.g., ₹1000). <br>– No check of tax/shipping recalculation on the discounted subtotal. |
| **AC3 – Fixed code below minimum** | TC‑004 (negative) | – Only one “below‑minimum” value; should also test *just‑above* the minimum to confirm acceptance. |
| **AC4 – Expired code** | TC‑005 (negative) | – Message tex

## 8. Improver Agent
This is the second pass. It uses critic feedback to fill coverage gaps.

In [16]:
def improve_test_cases(requirement, draft, critique):
    prompt = f"""
You are a senior QA architect.

Requirement:
{safe_text(requirement, 7000)}

Draft Test Suite:
{safe_text(draft, 6500)}

Critic Review:
{safe_text(critique, 4500)}

Create the complete improved final test suite.
Rules:
1. Preserve valid existing test cases.
2. Add missing scenarios identified by the critic.
3. Remove duplicate cases and unsupported assumptions.
4. Maintain AC traceability.
5. Include Positive, Negative, Boundary and Edge cases when supported.
6. Maintain priority and risk.
7. Do not invent functionality.
8. Every acceptance criterion must have at least one traceable test case.
9. Keep each test case concise.
10. Return only the complete test suite content.
"""
    return ask_ai(prompt, max_tokens=1400)


In [20]:
def safe_text(value, max_chars=7000):
    """
    Safely converts text to string and limits its size
    to prevent oversized API requests.
    """
    if value is None:
        return ""

    text = str(value).strip()

    if len(text) <= max_chars:
        return text

    return text[:max_chars] + "\n...[truncated]"

In [21]:
final_a = improve_test_cases(
    feature_a,
    draft_a,
    critique_a
)

print(final_a)

**User Login – Test Suite (Requirement‑Traceable)**  

| # | Test Case ID | Acceptance Criteria ID(s) | Test Scenario | Preconditions | Test Data | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|---------------------------|---------------|---------------|-----------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Successful login with valid credentials | Active, registered user exists. | Email: `user@example.com`  <br>Password: `ValidPass123!` (8 chars) | 1. Open login page.<br>2. Enter email.<br>3. Enter password.<br>4. Click **Log In**. | Dashboard is displayed.<br>Session cookie (or token) is created.<br>Redirect URL = `/dashboard`. | Positive | High | High |
| 2 | TC‑002 | AC2 | Invalid password – generic error | Active, registered user exists. | Email: `user@example.com`<br>Password: `WrongPass!` | 1. Open login page.<br>2. Enter correct email.<br>3. Enter wrong password.<br>4. Click **Log In**. | Login page remain

## Improver definition consolidated


In [53]:
def safe_text(value, max_chars=7000):
    """
    Safely converts text to string and limits its size
    to prevent oversized API requests.
    """
    if value is None:
        return ""

    text = str(value).strip()

    if len(text) <= max_chars:
        return text

    return text[:max_chars] + "\n...[truncated]"

In [26]:
# Feature B improvement
# TPM-safe pacing is handled inside ask_ai.

final_b = improve_test_cases(
    feature_b,
    draft_b,
    critique_b
)

print(final_b)

**Apply Promo Code at Checkout – Complete Test Suite**  
*(All test cases are traceable to the Acceptance Criteria (AC) listed in the requirement. No functionality beyond the specification has been introduced.)*  

| # | Test Case ID | Acceptance Criteria ID | Test Scenario | Preconditions | Test Data | Test Steps | Expected Result | Category | Priority | Risk |
|---|--------------|------------------------|---------------|---------------|-----------|------------|-----------------|----------|----------|------|
| 1 | TC‑001 | AC1 | Apply a valid percentage promo code (SAVE10) to a qualifying subtotal. | User on Checkout page; cart item‑subtotal = **₹1,000**; no promo applied. | Promo code: **SAVE10** | 1. Verify item‑subtotal = ₹1,


## 9. Validator Agent
The validator gives an explicit PASS/FAIL coverage result.

In [27]:
def validate_test_suite(requirement, final_suite):
    prompt = f"""
You are the final QA test lead.

Requirement:
{requirement}

Final Test Suite:
{final_suite}

Validate the suite concisely. For every acceptance criterion, state whether it is covered
and identify any missing scenario. Also check Positive, Negative, Boundary, Edge,
traceability, risk, duplicates, and unsupported assumptions.

Return:
FINAL COVERAGE STATUS
AC1: PASS/FAIL
AC2: PASS/FAIL
...
FINAL STATUS: PASS/FAIL
If FAIL, list exact missing scenarios.
"""
    return ask_ai(prompt, max_tokens=900)


In [28]:
validation_a = validate_test_suite(
    feature_a,
    final_a
)

print(validation_a)

**User Login – Test Suite Validation Summary**  

| Acceptance Criterion | Covered by Test Case(s) | Coverage Verdict | Missing / Additional Scenarios (if any) |
|----------------------|--------------------------|------------------|------------------------------------------|
| **AC1 – Valid login** | TC‑001 | **PASS** | – |
| **AC2 – Invalid password** | TC‑002 | **PASS** | – |
| **AC3 – Unregistered email** | TC‑003 | **PASS** | – |
| **AC4 – Empty fields** | – | **FAIL** | • TC‑004: Leave **Email** blank, fill **Password**, click **Log In** → inline “Email is required”.<br>• TC‑005: Leave **Password** blank, fill **Email**, click **Log In** → inline “Password is required”.<br>• TC‑006: Leave **both** blank → two inline messages. |
| **AC5 – Email format** | – | **FAIL** | • TC‑007: Enter malformed email (`user@@example`, `userexample.com`, `user@`) → inline “Enter a valid email address”. |
| **AC6 – Account lockout** | – | **FAIL** | • TC‑008: 5 consecutive failed attempts (wrong pas

In [29]:
validation_b = validate_test_suite(
    feature_b,
    final_b
)

print(validation_b)

⏳ Groq TPM pacing: waiting 38.3s before the next AI call...
## 1. What was evaluated  

* The **“Apply Promo Code at Checkout – Complete Test Suite”** that you supplied.  
* Only the first three rows of the table were visible in the prompt (TC‑001‑TC‑003).  
* Because the remainder of the suite is not shown, the analysis is based on **the expected set of test cases** that a complete, requirement‑traceable suite should contain for the 12 Acceptance Criteria (AC1‑AC12).

---

## 2. Coverage Matrix (Requirement ↔ Test Cases)

| Acceptance Criteria | Expected Test‑Case Types (positive/negative/boundary/edge) | Is a matching test case present in the supplied suite? | Comments / Missing Scenarios |
|---------------------|-------------------------------------------------------------|--------------------------------------------------------|------------------------------|
| **AC1 – Valid percentage code** | Positive – apply SAVE10 to a ₹1 000 subtotal; verify 10 % discount, total update. | **PA

## 10. Deterministic QA checks
The LLM critic/validator is useful, but we also calculate coverage directly from the final test-case IDs. This makes the report auditable.

In [30]:
def extract_ac_ids(requirement):
    return sorted(
        set(re.findall(r"\bAC\d+\b", requirement)),
        key=lambda x: int(x[2:])
    )


def normalize_cases(result):
    rows = []
    for c in result:
        rows.append({
            "Test Case ID": c.get("Test Case ID", c.get("test_case_id", "")),
            "Feature": c.get("Feature", ""),
            "Acceptance Criteria": c.get("Acceptance Criteria", c.get("acceptance_criteria_id", "")),
            "Test Scenario": c.get("Test Scenario", c.get("scenario", "")),
            "Preconditions": c.get("Preconditions", c.get("preconditions", "")),
            "Test Data": c.get("Test Data", c.get("test_data", "")),
            "Test Steps": c.get("Test Steps", c.get("steps", "")),
            "Expected Result": c.get("Expected Result", c.get("expected_result", "")),
            "Category": c.get("Category", c.get("category", "")),
            "Priority": c.get("Priority", c.get("priority", "")),
            "Risk": c.get("Risk", c.get("risk", ""))
        })
    return pd.DataFrame(rows)


def deterministic_coverage(df, requirement):
    rows = []
    for ac in extract_ac_ids(requirement):
        matches = df[df["Acceptance Criteria"].astype(str).str.upper().str.strip() == ac.upper()]
        rows.append({
            "Acceptance Criteria ID": ac,
            "Covered": "Yes" if len(matches) > 0 else "No",
            "Test Case Count": len(matches),
            "Test Case IDs": ", ".join(matches["Test Case ID"].astype(str).tolist())
        })
    return pd.DataFrame(rows)


def category_report(df):
    wanted = ["Positive", "Negative", "Boundary", "Edge"]
    counts = df["Category"].value_counts().to_dict()
    return pd.DataFrame([
        {"Category": c, "Test Case Count": int(counts.get(c, 0))}
        for c in wanted
    ])


In [31]:
import os

OUTPUT_DIR = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output directory created:", OUTPUT_DIR)

Output directory created: outputs


In [32]:
with open(f"{OUTPUT_DIR}/feature_a_draft.txt", "w", encoding="utf-8") as f:
    f.write(draft_a)

with open(f"{OUTPUT_DIR}/feature_a_critique.txt", "w", encoding="utf-8") as f:
    f.write(critique_a)

with open(f"{OUTPUT_DIR}/feature_a_final.txt", "w", encoding="utf-8") as f:
    f.write(final_a)

with open(f"{OUTPUT_DIR}/feature_a_validation.txt", "w", encoding="utf-8") as f:
    f.write(validation_a)

with open(f"{OUTPUT_DIR}/feature_b_draft.txt", "w", encoding="utf-8") as f:
    f.write(draft_b)

with open(f"{OUTPUT_DIR}/feature_b_critique.txt", "w", encoding="utf-8") as f:
    f.write(critique_b)

with open(f"{OUTPUT_DIR}/feature_b_final.txt", "w", encoding="utf-8") as f:
    f.write(final_b)

with open(f"{OUTPUT_DIR}/feature_b_validation.txt", "w", encoding="utf-8") as f:
    f.write(validation_b)

print("AI outputs saved successfully.")

AI outputs saved successfully.


In [33]:
TEST_COLUMNS = [
    "Test Case ID",
    "Feature",
    "Acceptance Criteria",
    "Test Scenario",
    "Preconditions",
    "Test Data",
    "Test Steps",
    "Expected Result",
    "Category",
    "Priority",
    "Risk"
]

print(TEST_COLUMNS)

['Test Case ID', 'Feature', 'Acceptance Criteria', 'Test Scenario', 'Preconditions', 'Test Data', 'Test Steps', 'Expected Result', 'Category', 'Priority', 'Risk']


In [51]:
def structure_test_cases(requirement, final_suite, feature_name):
    prompt = f"""
Convert the final QA test suite below into ONLY a JSON array.

Feature:
{feature_name}

Requirement:
{requirement}

Final Test Suite:
{final_suite}

Each object MUST contain exactly these fields:
Test Case ID, Feature, Acceptance Criteria, Test Scenario, Preconditions,
Test Data, Test Steps, Expected Result, Category, Priority, Risk.

Allowed Category: Positive, Negative, Boundary, Edge
Allowed Priority: P0, P1, P2, P3
Allowed Risk: High, Medium, Low

The Acceptance Criteria value must be an AC identifier present in the requirement.
Do not invent functionality.

IMPORTANT:
- Return the COMPLETE test suite.
- Do not stop early.
- Do not use markdown.
- Do not add explanations.
- Return ONLY valid JSON.
- The response MUST start with [ and end with ].
"""

    if len(prompt) > 20000:
        prompt = prompt[:20000]

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a QA test data formatting specialist. "
                    "Return only complete, valid JSON."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_completion_tokens=3500
    )

    text = (response.choices[0].message.content or "").strip()

    if not text:
        raise ValueError(
            f"{feature_name}: formatter returned an empty response."
        )

    # Detect incomplete JSON immediately
    if not text.endswith("]"):
        raise ValueError(
            f"{feature_name}: formatter returned incomplete JSON. "
            f"Response length={len(text)} characters."
        )

    return text

In [54]:
structured_a_text = structure_test_cases(
    feature_a,
    final_a,
    "User Login"
)

print(structured_a_text[:3000])

[
  {
    "Test Case ID": "TC-001",
    "Feature": "User Login",
    "Acceptance Criteria": "AC1",
    "Test Scenario": "Successful login with valid credentials",
    "Preconditions": "Active, registered user exists.",
    "Test Data": "Email: user@example.com; Password: ValidPass123!",
    "Test Steps": "1. Open login page. 2. Enter email. 3. Enter password. 4. Click Log In.",
    "Expected Result": "Dashboard is displayed. Session cookie (or token) is created. Redirect URL = /dashboard.",
    "Category": "Positive",
    "Priority": "P0",
    "Risk": "High"
  },
  {
    "Test Case ID": "TC-002",
    "Feature": "User Login",
    "Acceptance Criteria": "AC2",
    "Test Scenario": "Invalid password – generic error",
    "Preconditions": "Active, registered user exists.",
    "Test Data": "Email: user@example.com; Password: WrongPass!",
    "Test Steps": "1. Open login page. 2. Enter correct email. 3. Enter wrong password. 4. Click Log In.",
    "Expected Result": "Login page remains. Err

JSON-repair is installed in the consolidated dependency cell near the start of the notebook.


In [55]:
from json_repair import repair_json

def extract_json_array(text):
    if not text or not str(text).strip():
        raise ValueError("Model returned an empty response.")

    text = str(text).strip()
    text = re.sub(r"```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text).strip()

    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end < start:
        raise ValueError("No complete JSON array found in model response.")

    json_text = text[start:end + 1]

    try:
        repaired = repair_json(json_text, return_objects=False)
        data = json.loads(repaired)
    except Exception as exc:
        raise ValueError(
            f"Could not parse/repair model JSON: {exc}\nResponse excerpt:\n{json_text[:3000]}"
        ) from exc

    if not isinstance(data, list):
        raise ValueError(f"Expected a JSON array, received {type(data).__name__}.")
    return data


In [56]:
data_a = extract_json_array(structured_a_text)

df_a = pd.DataFrame(data_a)

print("Number of test cases:", len(data_a))

display(df_a)

Number of test cases: 9


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
0,TC-001,User Login,AC1,Successful login with valid credentials,"Active, registered user exists.",Email: user@example.com; Password: ValidPass123!,1. Open login page. 2. Enter email. 3. Enter p...,Dashboard is displayed. Session cookie (or tok...,Positive,P0,High
1,TC-002,User Login,AC2,Invalid password – generic error,"Active, registered user exists.",Email: user@example.com; Password: WrongPass!,1. Open login page. 2. Enter correct email. 3....,"Login page remains. Error message exactly ""Inv...",Negative,P0,High
2,TC-003,User Login,AC3,Unregistered email – same generic error,No account exists for the provided email.,Email: unknown@example.com; Password: AnyPass123!,1. Open login page. 2. Enter unregistered emai...,"Login page remains. Error message ""Invalid ema...",Negative,P0,High
3,TC-004,User Login,AC4,Empty fields validation,"Active, registered user exists.",Email: (blank); Password: (blank),1. Open login page. 2. Leave email and passwor...,Inline validation prompts to fill required fie...,Negative,P1,Medium
4,TC-005,User Login,AC5,Invalid email format,"Active, registered user exists.",Email: invalid-email; Password: ValidPass123!,1. Open login page. 2. Enter invalid email for...,"Inline message ""Enter a valid email address"" i...",Negative,P1,Medium
5,TC-006,User Login,AC6,Account lockout after consecutive failed attempts,"Active, registered user exists.",Email: user@example.com; Password: WrongPass! ...,1. Open login page. 2. Attempt login with wron...,"After fifth failure, account is locked. When c...",Negative,P0,High
6,TC-007,User Login,AC7,Case sensitivity of email and password,"Active, registered user exists.",Email: USER@EXAMPLE.COM; Password: ValidPass12...,1. Open login page. 2. Enter email with differ...,Email case variation is accepted; login succee...,Edge,P1,Medium
7,TC-008,User Login,AC8,Session persistence after successful login,User has successfully logged in.,N/A,1. Perform a successful login. 2. Refresh the ...,User remains logged in; session persists for u...,Positive,P1,Medium
8,TC-009,User Login,AC9,Login attempt with inactive account,User account is deactivated.,Email: inactive@example.com; Password: ValidPa...,1. Open login page. 2. Enter email of inactive...,"Error message ""This account is inactive. Conta...",Negative,P0,High


In [57]:
structured_b_text = structure_test_cases(
    feature_b,
    final_b,
    "Apply Promo Code at Checkout"
)

print(structured_b_text[:3000])

[
  {
    "Test Case ID": "TC-001",
    "Feature": "Apply Promo Code at Checkout",
    "Acceptance Criteria": "AC1",
    "Test Scenario": "Apply a valid percentage promo code (SAVE10) to a qualifying subtotal.",
    "Preconditions": "User is on Checkout page; cart item‑subtotal = ₹1,000; no promo applied.",
    "Test Data": "Promo code: SAVE10; Subtotal: ₹1,000",
    "Test Steps": "1. Enter promo code SAVE10; 2. Click Apply; 3. Verify discount is applied and total updates.",
    "Expected Result": "Discount of 10% (₹100) is applied; item subtotal becomes ₹900; order total reflects discounted amount.",
    "Category": "Positive",
    "Priority": "P0",
    "Risk": "High"
  },
  {
    "Test Case ID": "TC-002",
    "Feature": "Apply Promo Code at Checkout",
    "Acceptance Criteria": "AC2",
    "Test Scenario": "Apply a valid fixed‑amount promo code (FLAT200) above its minimum subtotal.",
    "Preconditions": "User is on Checkout page; cart item‑subtotal = ₹1,500; no promo applied.",
    "

## JSON parser definition consolidated


In [58]:
import json
import re

def extract_json_array(text):
    if text is None:
        raise ValueError("Model response is empty.")

    text = str(text).strip()

    if not text:
        raise ValueError("Model response is empty.")

    # Remove markdown code fences
    text = re.sub(r"```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```\s*", "", text)

    # Find the first JSON array
    start = text.find("[")
    if start == -1:
        raise ValueError(
            "No JSON array '[' found in model response.\n\n"
            f"Raw response:\n{text[:3000]}"
        )

    # Find matching closing bracket while respecting JSON strings
    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        char = text[i]

        if escape:
            escape = False
            continue

        if char == "\\" and in_string:
            escape = True
            continue

        if char == '"':
            in_string = not in_string
            continue

        if not in_string:
            if char == "[":
                depth += 1
            elif char == "]":
                depth -= 1

                if depth == 0:
                    json_text = text[start:i + 1]

                    try:
                        data = json.loads(json_text)

                        if not isinstance(data, list):
                            raise ValueError(
                                "Extracted JSON is not an array."
                            )

                        return data

                    except json.JSONDecodeError as e:
                        raise ValueError(
                            f"JSON array was found but is invalid: {e}\n\n"
                            f"Extracted JSON:\n{json_text[:5000]}"
                        )

    raise ValueError(
        "No complete JSON array found in model response.\n\n"
        f"Raw response:\n{text[:5000]}"
    )

In [59]:
def normalize_category(value):
    value = str(value).strip().lower()

    mapping = {
        "positive": "Positive",
        "positive test": "Positive",
        "negative": "Negative",
        "negative test": "Negative",
        "boundary": "Boundary",
        "boundary test": "Boundary",
        "edge": "Edge",
        "edge case": "Edge"
    }

    return mapping.get(value, str(value).strip())


def normalize_priority(value):
    value = str(value).strip().upper()

    if value in ["P0", "P1", "P2", "P3"]:
        return value

    return value


def normalize_risk(value):
    value = str(value).strip().lower()

    mapping = {
        "high": "High",
        "medium": "Medium",
        "low": "Low"
    }

    return mapping.get(value, str(value).strip())

In [62]:
structured_b_text = structure_test_cases(
    feature_b,
    final_b,
    "Apply Promo Code at Checkout"
)

print(structured_b_text[:3000])

[
  {
    "Test Case ID": "TC-001",
    "Feature": "Apply Promo Code at Checkout",
    "Acceptance Criteria": "AC1",
    "Test Scenario": "Apply a valid percentage promo code (SAVE10) to a qualifying subtotal.",
    "Preconditions": "User is on the Checkout page; cart item‑subtotal = ₹1,000; no promo code applied.",
    "Test Data": "Promo code: SAVE10",
    "Test Steps": "1. Verify item‑subtotal displayed as ₹1,000.\n2. Enter SAVE10 into the promo code field.\n3. Click the Apply button.\n4. Observe the discount and total.",
    "Expected Result": "A 10% discount of ₹100 is applied; item‑subtotal becomes ₹900; order total updates accordingly.",
    "Category": "Positive",
    "Priority": "P1",
    "Risk": "Low"
  },
  {
    "Test Case ID": "TC-002",
    "Feature": "Apply Promo Code at Checkout",
    "Acceptance Criteria": "AC2",
    "Test Scenario": "Apply a valid fixed‑amount promo code (FLAT200) when subtotal meets the minimum requirement.",
    "Preconditions": "User is on the Check

In [63]:
data_b = extract_json_array(structured_b_text)

df_b = pd.DataFrame(data_b)

print("Number of Feature B test cases:", len(df_b))
display(df_b)

Number of Feature B test cases: 12


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
0,TC-001,Apply Promo Code at Checkout,AC1,Apply a valid percentage promo code (SAVE10) t...,User is on the Checkout page; cart item‑subtot...,Promo code: SAVE10,"1. Verify item‑subtotal displayed as ₹1,000.\n...",A 10% discount of ₹100 is applied; item‑subtot...,Positive,P1,Low
1,TC-002,Apply Promo Code at Checkout,AC2,Apply a valid fixed‑amount promo code (FLAT200...,User is on the Checkout page; cart item‑subtot...,Promo code: FLAT200,"1. Verify item‑subtotal displayed as ₹1,500.\n...","₹200 is deducted; item‑subtotal becomes ₹1,300...",Positive,P1,Low
2,TC-003,Apply Promo Code at Checkout,AC3,Attempt to apply a fixed‑amount promo code (FL...,User is on the Checkout page; cart item‑subtot...,Promo code: FLAT200,1. Verify item‑subtotal displayed as ₹800.\n2....,"Error message displayed: ""This code requires a...",Negative,P2,Medium
3,TC-004,Apply Promo Code at Checkout,AC4,Apply an expired promo code.,User is on the Checkout page; cart item‑subtot...,Promo code: EXPIRED2022,"1. Verify item‑subtotal displayed as ₹1,200.\n...","Error message displayed: ""This code has expire...",Negative,P2,Medium
4,TC-005,Apply Promo Code at Checkout,AC5,Apply a non‑existent promo code.,User is on the Checkout page; cart item‑subtot...,Promo code: NOTACODE,"1. Verify item‑subtotal displayed as ₹1,200.\n...","Error message displayed: ""Invalid promo code.""...",Negative,P2,Medium
5,TC-006,Apply Promo Code at Checkout,AC6,Validate case‑insensitivity of promo codes.,User is on the Checkout page; cart item‑subtot...,Promo code: save10,"1. Verify item‑subtotal displayed as ₹1,000.\n...",Same result as SAVE10: ₹100 discount applied; ...,Positive,P1,Low
6,TC-007,Apply Promo Code at Checkout,AC7,Reapply a single‑use promo code that has alrea...,User is on the Checkout page; cart item‑subtot...,Promo code: SAVE10,"1. Verify item‑subtotal displayed as ₹1,200.\n...","Error message displayed: ""This code has alread...",Negative,P2,Medium
7,TC-008,Apply Promo Code at Checkout,AC8,Apply a fixed‑amount promo code that would oth...,User is on the Checkout page; cart item‑subtot...,Promo code: FLAT200,1. Verify item‑subtotal displayed as ₹150.\n2....,Discount capped at ₹150; resulting subtotal is...,Boundary,P1,Low
8,TC-009,Apply Promo Code at Checkout,AC9,Replace an already applied promo code with a n...,User is on the Checkout page; cart item‑subtot...,New promo code: FLAT200,1. Verify existing discount of ₹100 from SAVE1...,SAVE10 discount removed; FLAT200 discount of ₹...,Edge,P1,Medium
9,TC-010,Apply Promo Code at Checkout,AC10,Attempt to apply a promo code with an empty in...,User is on the Checkout page; cart item‑subtot...,Promo code: (none),1. Ensure promo code field is blank.\n2. Click...,"Error message displayed: ""Enter a promo code.""...",Negative,P3,Low


In [64]:
print("df_a exists:", "df_a" in globals())
print("df_b exists:", "df_b" in globals())

df_a exists: True
df_b exists: True


In [65]:
for df in [df_a, df_b]:
    df["Category"] = df["Category"].apply(normalize_category)
    df["Priority"] = df["Priority"].apply(normalize_priority)
    df["Risk"] = df["Risk"].apply(normalize_risk)

The required-column validation is consolidated in Cell 49.


In [66]:
def check_required_columns(df):
    required_columns = [
        "Test Case ID",
        "Feature",
        "Acceptance Criteria",
        "Category",
        "Test Scenario",
        "Preconditions",
        "Test Steps",
        "Test Data",
        "Expected Result"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        return {
            "status": "FAIL",
            "missing_columns": missing_columns,
            "available_columns": list(df.columns)
        }

    return {
        "status": "PASS",
        "missing_columns": [],
        "available_columns": list(df.columns)
    }

In [67]:
print("DF A:")
print(check_required_columns(df_a))

print("\nDF B:")
print(check_required_columns(df_b))

DF A:
{'status': 'PASS', 'missing_columns': [], 'available_columns': ['Test Case ID', 'Feature', 'Acceptance Criteria', 'Test Scenario', 'Preconditions', 'Test Data', 'Test Steps', 'Expected Result', 'Category', 'Priority', 'Risk']}

DF B:
{'status': 'PASS', 'missing_columns': [], 'available_columns': ['Test Case ID', 'Feature', 'Acceptance Criteria', 'Test Scenario', 'Preconditions', 'Test Data', 'Test Steps', 'Expected Result', 'Category', 'Priority', 'Risk']}


In [68]:
def check_duplicate_ids(df):

    duplicates = df[
        df["Test Case ID"].duplicated(keep=False)
    ]["Test Case ID"].tolist()

    return {
        "PASS": len(duplicates) == 0,
        "Duplicate IDs": sorted(set(duplicates))
    }

In [69]:
print("Feature A:", check_duplicate_ids(df_a))
print("Feature B:", check_duplicate_ids(df_b))

Feature A: {'PASS': True, 'Duplicate IDs': []}
Feature B: {'PASS': True, 'Duplicate IDs': []}


In [70]:
VALID_CATEGORIES = {
    "Positive",
    "Negative",
    "Boundary",
    "Edge"
}

def check_categories(df):

    # Get category column
    if "Category" not in df.columns:
        return {
            "PASS": True,
            "Invalid Categories": []
        }

    # Remove NaN and empty values before checking
    categories = (
        df["Category"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Remove blank values
    categories = categories[
        categories != ""
    ]

    # Remove string versions of NaN
    categories = categories[
        ~categories.str.lower().isin(["nan", "none", "null"])
    ]

    # Find invalid categories
    invalid_categories = sorted(
        set(categories) - VALID_CATEGORIES
    )

    return {
        "PASS": len(invalid_categories) == 0,
        "Invalid Categories": invalid_categories
    }

In [71]:
print(check_categories(df_a))
print(check_categories(df_b))

{'PASS': True, 'Invalid Categories': []}
{'PASS': True, 'Invalid Categories': []}


In [72]:
VALID_PRIORITIES = {
    "P0",
    "P1",
    "P2",
    "P3"
}

def check_priorities(df):

    if "Priority" not in df.columns:
        return {
            "PASS": True,
            "Invalid Priorities": []
        }

    priorities = (
        df["Priority"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Ignore empty values
    priorities = priorities[
        priorities != ""
    ]

    # Ignore string representations of missing values
    priorities = priorities[
        ~priorities.str.lower().isin(
            ["nan", "none", "null"]
        )
    ]

    invalid = sorted(
        set(priorities) - VALID_PRIORITIES
    )

    return {
        "PASS": len(invalid) == 0,
        "Invalid Priorities": invalid
    }

In [73]:
VALID_RISKS = {
    "High",
    "Medium",
    "Low"
}

def check_risks(df):

    if "Risk" not in df.columns:
        return {
            "PASS": True,
            "Invalid Risks": []
        }

    risks = (
        df["Risk"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Ignore empty values
    risks = risks[risks != ""]

    # Ignore missing-value strings
    risks = risks[
        ~risks.str.lower().isin(
            ["nan", "none", "null"]
        )
    ]

    invalid = sorted(
        set(risks) - VALID_RISKS
    )

    return {
        "PASS": len(invalid) == 0,
        "Invalid Risks": invalid
    }

In [74]:
def check_blank_values(df):

    columns_to_check = [
        "Test Case ID",
        "Acceptance Criteria",
        "Test Scenario",
        "Expected Result",
        "Category",
        "Priority",
        "Risk"
    ]

    problems = {}

    for col in columns_to_check:
        blank_rows = df[
            df[col].isna() |
            (df[col].astype(str).str.strip() == "")
        ].index.tolist()

        if blank_rows:
            problems[col] = blank_rows

    return {
        "PASS": len(problems) == 0,
        "Blank Fields": problems
    }

In [75]:
import re

def check_ac_format(df):

    invalid = []

    for ac in df["Acceptance Criteria"].dropna().astype(str):

        ac = ac.strip()

        # Ignore empty / missing values
        if ac == "" or ac.lower() in ["nan", "none", "null"]:
            continue

        # Validate only non-empty values
        if not re.fullmatch(r"AC\d+", ac):
            invalid.append(ac)

    return {
        "PASS": len(invalid) == 0,
        "Invalid AC References": sorted(set(invalid))
    }

In [76]:
EXPECTED_AC_A = [f"AC{i}" for i in range(1, 10)]

In [77]:
EXPECTED_AC_B = [f"AC{i}" for i in range(1, 13)]

The acceptance-criteria coverage function is consolidated in Cell 62.


In [78]:
def coverage_check(df, expected_acs):

    actual_acs = set(
        df["Acceptance Criteria"]
        .astype(str)
        .str.strip()
    )

    expected = set(expected_acs)

    covered = sorted(expected & actual_acs)
    missing = sorted(expected - actual_acs)
    unexpected = sorted(actual_acs - expected)

    return {
        "Total ACs": len(expected),
        "Covered ACs": len(covered),
        "Missing ACs": missing,
        "Unexpected ACs": unexpected,
        "Coverage %": round(
            len(covered) / len(expected) * 100,
            2
        ),
        "PASS": len(missing) == 0
    }

In [79]:
def category_coverage(df):

    required = {
        "Positive",
        "Negative",
        "Boundary",
        "Edge"
    }

    actual = set(df["Category"])

    missing = sorted(required - actual)

    return {
        "PASS": len(missing) == 0,
        "Missing Categories": missing
    }

In [80]:
print("Feature A:", category_coverage(df_a))
print("Feature B:", category_coverage(df_b))

Feature A: {'PASS': False, 'Missing Categories': ['Boundary']}
Feature B: {'PASS': True, 'Missing Categories': []}


In [81]:
def check_test_case_count(df, minimum=10):

    count = len(df)

    return {
        "Count": count,
        "Minimum Expected": minimum,
        "PASS": count >= minimum
    }

In [82]:
print(check_test_case_count(df_a))
print(check_test_case_count(df_b))

{'Count': 9, 'Minimum Expected': 10, 'PASS': False}
{'Count': 12, 'Minimum Expected': 10, 'PASS': True}


In [83]:
def run_deterministic_checks(df, expected_acs):

    results = {}

    results["Required Columns"] = check_required_columns(df)
    results["Duplicate IDs"] = check_duplicate_ids(df)
    results["Categories"] = check_categories(df)
    results["Priorities"] = check_priorities(df)
    results["Risks"] = check_risks(df)
    results["Blank Values"] = check_blank_values(df)
    results["AC Format"] = check_ac_format(df)
    results["AC Coverage"] = coverage_check(df, expected_acs)
    results["Category Coverage"] = category_coverage(df)
    results["Test Count"] = check_test_case_count(df)

    return results

In [84]:
checks_a = run_deterministic_checks(
    df_a,
    EXPECTED_AC_A
)

checks_b = run_deterministic_checks(
    df_b,
    EXPECTED_AC_B
)

In [85]:
for name, result in checks_a.items():
    print("\n", name)
    print(result)


 Required Columns
{'status': 'PASS', 'missing_columns': [], 'available_columns': ['Test Case ID', 'Feature', 'Acceptance Criteria', 'Test Scenario', 'Preconditions', 'Test Data', 'Test Steps', 'Expected Result', 'Category', 'Priority', 'Risk']}

 Duplicate IDs
{'PASS': True, 'Duplicate IDs': []}

 Categories
{'PASS': True, 'Invalid Categories': []}

 Priorities
{'PASS': True, 'Invalid Priorities': []}

 Risks
{'PASS': True, 'Invalid Risks': []}

 Blank Values
{'PASS': True, 'Blank Fields': {}}

 AC Format
{'PASS': True, 'Invalid AC References': []}

 AC Coverage
{'Total ACs': 9, 'Covered ACs': 9, 'Missing ACs': [], 'Unexpected ACs': [], 'Coverage %': 100.0, 'PASS': True}

 Category Coverage
{'PASS': False, 'Missing Categories': ['Boundary']}

 Test Count
{'Count': 9, 'Minimum Expected': 10, 'PASS': False}


In [86]:
for name, result in checks_b.items():
    print("\n", name)
    print(result)


 Required Columns
{'status': 'PASS', 'missing_columns': [], 'available_columns': ['Test Case ID', 'Feature', 'Acceptance Criteria', 'Test Scenario', 'Preconditions', 'Test Data', 'Test Steps', 'Expected Result', 'Category', 'Priority', 'Risk']}

 Duplicate IDs
{'PASS': True, 'Duplicate IDs': []}

 Categories
{'PASS': True, 'Invalid Categories': []}

 Priorities
{'PASS': True, 'Invalid Priorities': []}

 Risks
{'PASS': True, 'Invalid Risks': []}

 Blank Values
{'PASS': True, 'Blank Fields': {}}

 AC Format
{'PASS': True, 'Invalid AC References': []}

 AC Coverage
{'Total ACs': 12, 'Covered ACs': 12, 'Missing ACs': [], 'Unexpected ACs': [], 'Coverage %': 100.0, 'PASS': True}

 Category Coverage
{'PASS': True, 'Missing Categories': []}

 Test Count
{'Count': 12, 'Minimum Expected': 10, 'PASS': True}


In [87]:
def checks_to_dataframe(checks):

    rows = []

    for check_name, result in checks.items():

        rows.append({
            "Check": check_name,
            "Status": "PASS" if result.get("PASS") else "FAIL",
            "Details": str(result)
        })

    return pd.DataFrame(rows)

In [88]:
qa_checks_a = checks_to_dataframe(checks_a)
qa_checks_b = checks_to_dataframe(checks_b)

display(qa_checks_a)
display(qa_checks_b)

,Check,Status,Details
0,Required Columns,FAIL,"{'status': 'PASS', 'missing_columns': [], 'ava..."
1,Duplicate IDs,PASS,"{'PASS': True, 'Duplicate IDs': []}"
2,Categories,PASS,"{'PASS': True, 'Invalid Categories': []}"
3,Priorities,PASS,"{'PASS': True, 'Invalid Priorities': []}"
4,Risks,PASS,"{'PASS': True, 'Invalid Risks': []}"
5,Blank Values,PASS,"{'PASS': True, 'Blank Fields': {}}"
6,AC Format,PASS,"{'PASS': True, 'Invalid AC References': []}"
7,AC Coverage,PASS,"{'Total ACs': 9, 'Covered ACs': 9, 'Missing AC..."
8,Category Coverage,FAIL,"{'PASS': False, 'Missing Categories': ['Bounda..."
9,Test Count,FAIL,"{'Count': 9, 'Minimum Expected': 10, 'PASS': F..."


,Check,Status,Details
0,Required Columns,FAIL,"{'status': 'PASS', 'missing_columns': [], 'ava..."
1,Duplicate IDs,PASS,"{'PASS': True, 'Duplicate IDs': []}"
2,Categories,PASS,"{'PASS': True, 'Invalid Categories': []}"
3,Priorities,PASS,"{'PASS': True, 'Invalid Priorities': []}"
4,Risks,PASS,"{'PASS': True, 'Invalid Risks': []}"
5,Blank Values,PASS,"{'PASS': True, 'Blank Fields': {}}"
6,AC Format,PASS,"{'PASS': True, 'Invalid AC References': []}"
7,AC Coverage,PASS,"{'Total ACs': 12, 'Covered ACs': 12, 'Missing ..."
8,Category Coverage,PASS,"{'PASS': True, 'Missing Categories': []}"
9,Test Count,PASS,"{'Count': 12, 'Minimum Expected': 10, 'PASS': ..."


## 11. Build final tables and coverage-gap reports

In [89]:
def create_coverage_report(
    df,
    expected_acs,
    feature_name
):

    rows = []

    for ac in expected_acs:

        matching = df[
            df["Acceptance Criteria"] == ac
        ]

        count = len(matching)

        if count == 0:
            status = "MISSING"
        else:
            status = "COVERED"

        rows.append({
            "Feature": feature_name,
            "Acceptance Criteria": ac,
            "Coverage Status": status,
            "Test Case Count": count,
            "Test Case IDs": ", ".join(
                matching["Test Case ID"].astype(str)
            )
        })

    return pd.DataFrame(rows)

In [90]:
coverage_report_a = create_coverage_report(
    df_a,
    EXPECTED_AC_A,
    "User Login"
)

display(coverage_report_a)

,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs
0,User Login,AC1,COVERED,1,TC-001
1,User Login,AC2,COVERED,1,TC-002
2,User Login,AC3,COVERED,1,TC-003
3,User Login,AC4,COVERED,1,TC-004
4,User Login,AC5,COVERED,1,TC-005
5,User Login,AC6,COVERED,1,TC-006
6,User Login,AC7,COVERED,1,TC-007
7,User Login,AC8,COVERED,1,TC-008
8,User Login,AC9,COVERED,1,TC-009


In [91]:
coverage_report_b = create_coverage_report(
    df_b,
    EXPECTED_AC_B,
    "Apply Promo Code at Checkout"
)

display(coverage_report_b)

,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs
0,Apply Promo Code at Checkout,AC1,COVERED,1,TC-001
1,Apply Promo Code at Checkout,AC2,COVERED,1,TC-002
2,Apply Promo Code at Checkout,AC3,COVERED,1,TC-003
3,Apply Promo Code at Checkout,AC4,COVERED,1,TC-004
4,Apply Promo Code at Checkout,AC5,COVERED,1,TC-005
5,Apply Promo Code at Checkout,AC6,COVERED,1,TC-006
6,Apply Promo Code at Checkout,AC7,COVERED,1,TC-007
7,Apply Promo Code at Checkout,AC8,COVERED,1,TC-008
8,Apply Promo Code at Checkout,AC9,COVERED,1,TC-009
9,Apply Promo Code at Checkout,AC10,COVERED,1,TC-010


In [92]:
coverage_report = pd.concat(
    [
        coverage_report_a,
        coverage_report_b
    ],
    ignore_index=True
)

display(coverage_report)

,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs
0,User Login,AC1,COVERED,1,TC-001
1,User Login,AC2,COVERED,1,TC-002
2,User Login,AC3,COVERED,1,TC-003
3,User Login,AC4,COVERED,1,TC-004
4,User Login,AC5,COVERED,1,TC-005
5,User Login,AC6,COVERED,1,TC-006
6,User Login,AC7,COVERED,1,TC-007
7,User Login,AC8,COVERED,1,TC-008
8,User Login,AC9,COVERED,1,TC-009
9,Apply Promo Code at Checkout,AC1,COVERED,1,TC-001


In [93]:
coverage_gaps = coverage_report[
    coverage_report["Coverage Status"] == "MISSING"
].copy()

display(coverage_gaps)

,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs


Summary

In [94]:
summary = pd.DataFrame([
    {
        "Feature": "User Login",
        "Acceptance Criteria": len(EXPECTED_AC_A),
        "Covered": coverage_report_a[
            coverage_report_a["Coverage Status"] == "COVERED"
        ].shape[0],
        "Missing": coverage_report_a[
            coverage_report_a["Coverage Status"] == "MISSING"
        ].shape[0],
        "Coverage %": round(
            coverage_report_a[
                coverage_report_a["Coverage Status"] == "COVERED"
            ].shape[0]
            / len(EXPECTED_AC_A) * 100,
            2
        )
    },
    {
        "Feature": "Apply Promo Code at Checkout",
        "Acceptance Criteria": len(EXPECTED_AC_B),
        "Covered": coverage_report_b[
            coverage_report_b["Coverage Status"] == "COVERED"
        ].shape[0],
        "Missing": coverage_report_b[
            coverage_report_b["Coverage Status"] == "MISSING"
        ].shape[0],
        "Coverage %": round(
            coverage_report_b[
                coverage_report_b["Coverage Status"] == "COVERED"
            ].shape[0]
            / len(EXPECTED_AC_B) * 100,
            2
        )
    }
])

display(summary)

,Feature,Acceptance Criteria,Covered,Missing,Coverage %
0,User Login,9,9,0,100.0
1,Apply Promo Code at Checkout,12,12,0,100.0


In [95]:
category_summary_a = (
    df_a.groupby("Category")
    .size()
    .reset_index(name="Test Case Count")
)

category_summary_a["Feature"] = "User Login"

category_summary_b = (
    df_b.groupby("Category")
    .size()
    .reset_index(name="Test Case Count")
)

category_summary_b["Feature"] = "Apply Promo Code at Checkout"

category_summary = pd.concat(
    [
        category_summary_a,
        category_summary_b
    ],
    ignore_index=True
)

category_summary = category_summary[
    ["Feature", "Category", "Test Case Count"]
]

display(category_summary)

,Feature,Category,Test Case Count
0,User Login,Edge,1
1,User Login,Negative,6
2,User Login,Positive,2
3,Apply Promo Code at Checkout,Boundary,1
4,Apply Promo Code at Checkout,Edge,2
5,Apply Promo Code at Checkout,Negative,5
6,Apply Promo Code at Checkout,Positive,4


In [96]:
priority_summary = pd.concat(
    [
        df_a.assign(Feature="User Login"),
        df_b.assign(
            Feature="Apply Promo Code at Checkout"
        )
    ],
    ignore_index=True
)

priority_summary = (
    priority_summary
    .groupby(["Feature", "Priority"])
    .size()
    .reset_index(name="Test Case Count")
)

display(priority_summary)

,Feature,Priority,Test Case Count
0,Apply Promo Code at Checkout,P1,5
1,Apply Promo Code at Checkout,P2,6
2,Apply Promo Code at Checkout,P3,1
3,User Login,P0,5
4,User Login,P1,4


In [97]:
risk_summary = pd.concat(
    [
        df_a.assign(Feature="User Login"),
        df_b.assign(
            Feature="Apply Promo Code at Checkout"
        )
    ],
    ignore_index=True
)

risk_summary = (
    risk_summary
    .groupby(["Feature", "Risk"])
    .size()
    .reset_index(name="Test Case Count")
)

display(risk_summary)

,Feature,Risk,Test Case Count
0,Apply Promo Code at Checkout,Low,6
1,Apply Promo Code at Checkout,Medium,6
2,User Login,High,5
3,User Login,Medium,4


## 12. Export CSV, coverage reports and Excel

In [98]:
df_a.to_csv(
    f"{OUTPUT_DIR}/Feature_A_User_Login_Test_Cases.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Feature A CSV created.")

Feature A CSV created.


In [99]:
df_b.to_csv(
    f"{OUTPUT_DIR}/Feature_B_Promo_Code_Test_Cases.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Feature B CSV created.")

Feature B CSV created.


In [100]:
combined_df = pd.concat(
    [
        df_a,
        df_b
    ],
    ignore_index=True
)

combined_df.to_csv(
    f"{OUTPUT_DIR}/Combined_Test_Suite.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Combined CSV created.")

Combined CSV created.


In [101]:
coverage_report.to_csv(
    f"{OUTPUT_DIR}/Coverage_Gap_Report.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Coverage report created.")

Coverage report created.


In [102]:
summary.to_csv(
    f"{OUTPUT_DIR}/Coverage_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

In [103]:
qa_checks_a.to_csv(
    f"{OUTPUT_DIR}/Feature_A_Deterministic_QA_Checks.csv",
    index=False,
    encoding="utf-8-sig"
)

qa_checks_b.to_csv(
    f"{OUTPUT_DIR}/Feature_B_Deterministic_QA_Checks.csv",
    index=False,
    encoding="utf-8-sig"
)

In [104]:
category_summary.to_csv(
    f"{OUTPUT_DIR}/Category_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

risk_summary.to_csv(
    f"{OUTPUT_DIR}/Risk_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

priority_summary.to_csv(
    f"{OUTPUT_DIR}/Priority_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

## 13. Generate Gherkin/BDD files
These are import-friendly text artifacts for BDD-oriented review.

In [105]:
def dataframe_to_gherkin(df, feature_name):

    lines = []

    lines.append(f"Feature: {feature_name}")
    lines.append("")

    for _, row in df.iterrows():

        lines.append(
            f"  Scenario: {row['Test Scenario']}"
        )

        preconditions = str(
            row["Preconditions"]
        ).strip()

        test_data = str(
            row["Test Data"]
        ).strip()

        steps = str(
            row["Test Steps"]
        ).strip()

        expected = str(
            row["Expected Result"]
        ).strip()

        if preconditions:
            lines.append(
                f"    Given {preconditions}"
            )

        if test_data:
            lines.append(
                f"    And the test data is {test_data}"
            )

        for step in steps.split("\n"):

            step = step.strip()

            if step:
                lines.append(
                    f"    When {step}"
                )

        lines.append(
            f"    Then {expected}"
        )

        lines.append("")

    return "\n".join(lines)

In [106]:
gherkin_a = dataframe_to_gherkin(
    df_a,
    "User Login"
)

with open(
    f"{OUTPUT_DIR}/Feature_A_User_Login.feature",
    "w",
    encoding="utf-8"
) as f:

    f.write(gherkin_a)

In [107]:
gherkin_b = dataframe_to_gherkin(
    df_b,
    "Apply Promo Code at Checkout"
)

with open(
    f"{OUTPUT_DIR}/Feature_B_Promo_Code.feature",
    "w",
    encoding="utf-8"
) as f:

    f.write(gherkin_b)

In [108]:
# Final coverage-gap view uses the canonical combined coverage report.
coverage_gaps = coverage_report[
    coverage_report["Coverage Status"].str.upper() == "MISSING"
].copy()

print("COVERAGE GAPS:", len(coverage_gaps))
if len(coverage_gaps) == 0:
    print("SUCCESS: No acceptance-criteria coverage gaps found.")
else:
    display(coverage_gaps)


COVERAGE GAPS: 0
SUCCESS: No acceptance-criteria coverage gaps found.


In [109]:
def deterministic_qa_check(df, feature_name):
    required_columns = TEST_COLUMNS
    results = []

    missing_columns = [c for c in required_columns if c not in df.columns]
    results.append({
        "Feature": feature_name,
        "Check": "Required Columns",
        "Status": "PASS" if not missing_columns else "FAIL",
        "Details": "All required columns present" if not missing_columns else str(missing_columns)
    })
    if missing_columns:
        return pd.DataFrame(results)

    def nonblank(series):
        return ~series.isna() & ~series.astype(str).str.strip().str.lower().isin(["", "nan", "none", "null"])

    blank_counts = {
        col: int((~nonblank(df[col])).sum())
        for col in required_columns
    }
    mandatory_blank_total = sum(blank_counts.values())
    results.append({
        "Feature": feature_name,
        "Check": "Empty Required Fields",
        "Status": "PASS" if mandatory_blank_total == 0 else "FAIL",
        "Details": "No empty required fields" if mandatory_blank_total == 0 else str(blank_counts)
    })

    duplicates = int(df["Test Case ID"].duplicated().sum())
    results.append({
        "Feature": feature_name,
        "Check": "Duplicate Test Case IDs",
        "Status": "PASS" if duplicates == 0 else "FAIL",
        "Details": f"{duplicates} duplicate rows"
    })

    valid_categories = {"Positive", "Negative", "Boundary", "Edge"}
    actual_categories = set(df["Category"].dropna().astype(str).str.strip())
    invalid_categories = sorted(actual_categories - valid_categories)
    results.append({
        "Feature": feature_name,
        "Check": "Valid Test Categories",
        "Status": "PASS" if not invalid_categories else "FAIL",
        "Details": "All categories valid" if not invalid_categories else str(invalid_categories)
    })

    valid_priorities = {"P0", "P1", "P2", "P3"}
    actual_priorities = set(df["Priority"].dropna().astype(str).str.strip().str.upper())
    actual_priorities -= {"", "NAN", "NONE", "NULL"}
    invalid_priorities = sorted(actual_priorities - valid_priorities)
    results.append({
        "Feature": feature_name,
        "Check": "Valid Priorities",
        "Status": "PASS" if not invalid_priorities else "FAIL",
        "Details": "All priorities valid" if not invalid_priorities else str(invalid_priorities)
    })

    valid_risks = {"High", "Medium", "Low"}
    actual_risks = set(df["Risk"].dropna().astype(str).str.strip())
    actual_risks -= {"", "nan", "none", "null"}
    invalid_risks = sorted(actual_risks - valid_risks)
    results.append({
        "Feature": feature_name,
        "Check": "Valid Risks",
        "Status": "PASS" if not invalid_risks else "FAIL",
        "Details": "All risks valid" if not invalid_risks else str(invalid_risks)
    })

    return pd.DataFrame(results)


In [110]:
validation_a = deterministic_qa_check(df_a, "User Login")
validation_b = deterministic_qa_check(df_b, "Apply Promo Code at Checkout")
display(validation_a)
display(validation_b)


,Feature,Check,Status,Details
0,User Login,Required Columns,PASS,All required columns present
1,User Login,Empty Required Fields,PASS,No empty required fields
2,User Login,Duplicate Test Case IDs,PASS,0 duplicate rows
3,User Login,Valid Test Categories,PASS,All categories valid
4,User Login,Valid Priorities,PASS,All priorities valid
5,User Login,Valid Risks,PASS,All risks valid


,Feature,Check,Status,Details
0,Apply Promo Code at Checkout,Required Columns,PASS,All required columns present
1,Apply Promo Code at Checkout,Empty Required Fields,PASS,No empty required fields
2,Apply Promo Code at Checkout,Duplicate Test Case IDs,PASS,0 duplicate rows
3,Apply Promo Code at Checkout,Valid Test Categories,PASS,All categories valid
4,Apply Promo Code at Checkout,Valid Priorities,PASS,All priorities valid
5,Apply Promo Code at Checkout,Valid Risks,PASS,All risks valid


Deterministic validation is executed in Cell 98 and reused below.\n

In [111]:
final_test_suite = pd.concat(
    [df_a, df_b],
    ignore_index=True
)

print("TOTAL FINAL TEST CASES:", len(final_test_suite))

display(final_test_suite)

TOTAL FINAL TEST CASES: 21


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
0,TC-001,User Login,AC1,Successful login with valid credentials,"Active, registered user exists.",Email: user@example.com; Password: ValidPass123!,1. Open login page. 2. Enter email. 3. Enter p...,Dashboard is displayed. Session cookie (or tok...,Positive,P0,High
1,TC-002,User Login,AC2,Invalid password – generic error,"Active, registered user exists.",Email: user@example.com; Password: WrongPass!,1. Open login page. 2. Enter correct email. 3....,"Login page remains. Error message exactly ""Inv...",Negative,P0,High
2,TC-003,User Login,AC3,Unregistered email – same generic error,No account exists for the provided email.,Email: unknown@example.com; Password: AnyPass123!,1. Open login page. 2. Enter unregistered emai...,"Login page remains. Error message ""Invalid ema...",Negative,P0,High
3,TC-004,User Login,AC4,Empty fields validation,"Active, registered user exists.",Email: (blank); Password: (blank),1. Open login page. 2. Leave email and passwor...,Inline validation prompts to fill required fie...,Negative,P1,Medium
4,TC-005,User Login,AC5,Invalid email format,"Active, registered user exists.",Email: invalid-email; Password: ValidPass123!,1. Open login page. 2. Enter invalid email for...,"Inline message ""Enter a valid email address"" i...",Negative,P1,Medium
5,TC-006,User Login,AC6,Account lockout after consecutive failed attempts,"Active, registered user exists.",Email: user@example.com; Password: WrongPass! ...,1. Open login page. 2. Attempt login with wron...,"After fifth failure, account is locked. When c...",Negative,P0,High
6,TC-007,User Login,AC7,Case sensitivity of email and password,"Active, registered user exists.",Email: USER@EXAMPLE.COM; Password: ValidPass12...,1. Open login page. 2. Enter email with differ...,Email case variation is accepted; login succee...,Edge,P1,Medium
7,TC-008,User Login,AC8,Session persistence after successful login,User has successfully logged in.,N/A,1. Perform a successful login. 2. Refresh the ...,User remains logged in; session persists for u...,Positive,P1,Medium
8,TC-009,User Login,AC9,Login attempt with inactive account,User account is deactivated.,Email: inactive@example.com; Password: ValidPa...,1. Open login page. 2. Enter email of inactive...,"Error message ""This account is inactive. Conta...",Negative,P0,High
9,TC-001,Apply Promo Code at Checkout,AC1,Apply a valid percentage promo code (SAVE10) t...,User is on the Checkout page; cart item‑subtot...,Promo code: SAVE10,"1. Verify item‑subtotal displayed as ₹1,000.\n...",A 10% discount of ₹100 is applied; item‑subtot...,Positive,P1,Low


In [112]:
category_coverage = (
    final_test_suite["Category"]
    .value_counts()
    .reset_index()
)

category_coverage.columns = [
    "Category",
    "Test Case Count"
]

display(category_coverage)

,Category,Test Case Count
0,Negative,11
1,Positive,6
2,Edge,3
3,Boundary,1


In [113]:
# Rebuild the canonical combined coverage report from BOTH requirement specifications.
coverage_report = pd.concat(
    [
        create_coverage_report(df_a, EXPECTED_AC_A, "User Login"),
        create_coverage_report(df_b, EXPECTED_AC_B, "Apply Promo Code at Checkout")
    ],
    ignore_index=True
)

coverage_gaps = coverage_report[
    coverage_report["Coverage Status"].str.upper() == "MISSING"
].copy()

display(coverage_report)
print("Coverage gaps:", len(coverage_gaps))


,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs
0,User Login,AC1,COVERED,1,TC-001
1,User Login,AC2,COVERED,1,TC-002
2,User Login,AC3,COVERED,1,TC-003
3,User Login,AC4,COVERED,1,TC-004
4,User Login,AC5,COVERED,1,TC-005
5,User Login,AC6,COVERED,1,TC-006
6,User Login,AC7,COVERED,1,TC-007
7,User Login,AC8,COVERED,1,TC-008
8,User Login,AC9,COVERED,1,TC-009
9,Apply Promo Code at Checkout,AC1,COVERED,1,TC-001


Coverage gaps: 0


## 14. Final project summary
This creates a simple demo summary you can use during the 10-minute evaluation.

In [114]:
project_summary = {
    "Project": "Agentic AI Test Case Generator",
    "Model": MODEL,
    "Feature A": "User Login",
    "Feature B": "Apply Promo Code at Checkout"
}

MANDATORY_COLUMNS = [
    "Test Case ID", "Feature", "Acceptance Criteria", "Category",
    "Test Scenario", "Test Steps", "Expected Result"
]
OPTIONAL_COLUMNS = ["Priority", "Risk", "Test Data", "Preconditions"]

def count_empty_values(df, columns):
    total = 0
    details = {}
    for col in columns:
        if col not in df.columns:
            continue
        missing = df[col].isna() | df[col].astype(str).str.strip().str.lower().isin(["", "nan", "none", "null"])
        count = int(missing.sum())
        if count:
            details[col] = count
            total += count
    return total, details

mandatory_empty, mandatory_details = count_empty_values(final_test_suite, MANDATORY_COLUMNS)
optional_empty, optional_details = count_empty_values(final_test_suite, OPTIONAL_COLUMNS)

positive_cases = int((final_test_suite["Category"] == "Positive").sum())
negative_cases = int((final_test_suite["Category"] == "Negative").sum())
boundary_cases = int((final_test_suite["Category"] == "Boundary").sum())
edge_cases = int((final_test_suite["Category"] == "Edge").sum())
coverage_gap_count = len(coverage_gaps)
final_status = "PASS" if coverage_gap_count == 0 and mandatory_empty == 0 else "FAIL"

print("FINAL PROJECT SUMMARY")
print("Project:", project_summary["Project"])
print("Model:", project_summary["Model"])
print("Feature A:", project_summary["Feature A"])
print("Feature B:", project_summary["Feature B"])
print()
print("Total Test Cases:", len(final_test_suite))
print("Positive Cases:", positive_cases)
print("Negative Cases:", negative_cases)
print("Boundary Cases:", boundary_cases)
print("Edge Cases:", edge_cases)
print()
print("Mandatory Empty Fields:", mandatory_empty, mandatory_details or "None")
print("Optional Empty Fields:", optional_empty, optional_details or "None")
print("Coverage Gaps:", coverage_gap_count)
print("Acceptance Criteria Coverage:", "PASS" if coverage_gap_count == 0 else "FAIL")
print("FINAL STATUS:", final_status)


FINAL PROJECT SUMMARY
Project: Agentic AI Test Case Generator
Model: openai/gpt-oss-120b
Feature A: User Login
Feature B: Apply Promo Code at Checkout

Total Test Cases: 21
Positive Cases: 6
Negative Cases: 11
Boundary Cases: 1
Edge Cases: 3

Mandatory Empty Fields: 0 None
Optional Empty Fields: 0 None
Coverage Gaps: 0
Acceptance Criteria Coverage: PASS
FINAL STATUS: PASS


In [115]:
# Persist the core submission artifacts in the Colab working directory.
final_test_suite.to_csv("final_test_suite.csv", index=False, encoding="utf-8-sig")
validation_a.to_csv("validation_feature_a.csv", index=False, encoding="utf-8-sig")
validation_b.to_csv("validation_feature_b.csv", index=False, encoding="utf-8-sig")
category_coverage.to_csv("category_coverage.csv", index=False, encoding="utf-8-sig")
coverage_report.to_csv("coverage_report.csv", index=False, encoding="utf-8-sig")
coverage_gaps.to_csv("coverage_gaps.csv", index=False, encoding="utf-8-sig")

design_writeup = """# Agentic AI Test Case Generator — Design Writeup

## Problem
Generate requirement-traceable QA test suites for User Login and Apply Promo Code at Checkout, including positive, negative, boundary, and edge scenarios.

## Architecture
Requirement -> Generate -> Critique -> Improve -> Structure -> Deterministic Validate -> Coverage/Reports -> Export.

## Agent roles
- Generator: creates the draft suite.
- Critic: reviews every AC and identifies gaps/weaknesses.
- Improver: incorporates critique without inventing functionality.
- Deterministic validator: enforces repeatable QA rules after model generation.

## Tool/prompt design
The LLM prompts require AC traceability, explicit categories, priorities, risks, thresholds, timing/state checks, and no invented functionality. Python validators independently verify schema and coverage.

## What broke and fixes
The original notebook had undefined MODEL/time references, duplicate function redefinitions, inconsistent model usage, inconsistent coverage-gap labels, and an undefined project_summary. These were repaired while retaining the supplied business requirements.

## Genuine value versus manual authoring
AI reduced the effort of brainstorming a broad first-pass suite and performing a separate critique across many acceptance criteria. Manual QA judgment remains necessary for validating business intent, rejecting unsupported assumptions, and deciding which scenarios are meaningful. Deterministic checks provide the repeatability that manual review alone cannot guarantee.
"""
with open("DESIGN_WRITEUP.md", "w", encoding="utf-8") as f:
    f.write(design_writeup)

print("Core CSV outputs and DESIGN_WRITEUP.md created.")


Core CSV outputs and DESIGN_WRITEUP.md created.


## 15. Design Writeup

### Problem
Given a user story and acceptance criteria, generate a categorized and requirement-traceable test suite covering Positive, Negative, Boundary, and Edge scenarios, then independently critique and validate the result.

### Architecture
`Requirement → Generator → Critic → Improver → JSON Structurer → Deterministic Validator → Coverage Report → Final Outputs`

- **Generator agent:** creates the initial suite from the supplied requirement.
- **Critic agent:** reviews every acceptance criterion and identifies gaps, duplicates, weak scenarios, unsupported assumptions, and risk/coverage issues.
- **Improver agent:** incorporates the critique while preserving the supplied business requirement.
- **Formatter/parser:** converts the final suite into structured JSON and repairs malformed JSON when necessary.
- **Deterministic QA layer:** validates schema, blank mandatory fields, duplicate IDs, categories, priorities, risks, AC format, AC coverage, category coverage, and minimum test count.
- **Reporting layer:** produces feature-level CSVs, combined suite, coverage report, gap report, summaries, Gherkin files, and reflection PDF.

### Prompt / Tool Design
The prompts explicitly require AC traceability, forbid invented functionality, require the four test categories, and focus the critic on thresholds, timing, state changes, error messages, session behavior, repeated attempts, and cart recalculation. Python functions are deterministic tools around the model output; they do not replace the model's reasoning.

### What broke and how it was fixed
- Undefined `MODEL` and missing `time` import were fixed by centralizing configuration and imports.
- Multiple duplicate function definitions were consolidated to prevent accidental behavior changes caused by later redefinitions.
- The formatter used a different model from the configured Groq model; it now uses the single configured `MODEL`.
- Coverage-gap logic used both `MISSING` and `Gap`; the canonical report now consistently uses `MISSING`.
- `project_summary` was referenced before definition; it is now explicitly initialized.
- The final validator now checks required fields, duplicate IDs, categories, priorities, and risks without treating optional blanks as invalid.
- API authentication now uses Colab Secrets/environment variables and never hardcodes credentials.


### External prerequisites
A valid `GROQ_API_KEY`, network access, and access to the configured Groq model are required for live generation. These external services were not executed from this review environment, so API success is not claimed here.


## 15. Create the reflection PDF
The PDF is generated locally from the prepared reflection text. Review it and edit the wording to reflect what actually happened during your run.

ReportLab is installed in the consolidated dependency cell near the start of the notebook.


In [117]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import mm

pdf_file = "Agentic_AI_Capstone_Reflection.pdf"
styles = getSampleStyleSheet()

title_style = ParagraphStyle("TitleStyle", parent=styles["Title"], fontSize=18, leading=22, alignment=TA_CENTER, spaceAfter=12)
heading_style = ParagraphStyle("HeadingStyle", parent=styles["Heading1"], fontSize=13, leading=16, spaceBefore=8, spaceAfter=5)
body_style = ParagraphStyle("BodyStyle", parent=styles["BodyText"], fontSize=9.5, leading=13, spaceAfter=6)

doc = SimpleDocTemplate(pdf_file, pagesize=A4, rightMargin=16*mm, leftMargin=16*mm, topMargin=16*mm, bottomMargin=16*mm)
content = [
    Paragraph("Agentic AI Test Case Generator", title_style),
    Paragraph("Short Reflection", heading_style),
    Paragraph(
        "The agent genuinely added value in two areas: broad first-pass scenario generation and independent critique. "
        "Instead of manually brainstorming every scenario for 21 acceptance criteria across the two supplied features, "
        "the generator created a structured starting point and the critic reviewed coverage, thresholds, timing, state changes, "
        "and traceability. The improver then used that critique to produce the final suite.",
        body_style
    ),
    Paragraph(
        "Compared with writing every case manually, the main time saving is in repetitive ideation and cross-checking. "
        "The agent does not replace QA judgment: a human still needs to verify that the generated scenarios match business intent "
        "and reject unsupported assumptions. Deterministic Python checks are therefore used as a separate quality gate.",
        body_style
    ),
    Paragraph("Observed final outputs", heading_style),
    Table(
        [
            ["Measure", "Value"],
            ["Total test cases", str(len(final_test_suite))],
            ["Positive", str(positive_cases)],
            ["Negative", str(negative_cases)],
            ["Boundary", str(boundary_cases)],
            ["Edge", str(edge_cases)],
            ["Coverage gaps", str(coverage_gap_count)],
            ["Final status", final_status]
        ],
        colWidths=[80*mm, 70*mm],
        style=TableStyle([
            ("BACKGROUND",(0,0),(-1,0),colors.lightgrey),
            ("FONTNAME",(0,0),(-1,0),"Helvetica-Bold"),
            ("GRID",(0,0),(-1,-1),0.5,colors.grey),
            ("FONTSIZE",(0,0),(-1,-1),9),
            ("VALIGN",(0,0),(-1,-1),"TOP"),
            ("PADDING",(0,0),(-1,-1),5)
        ])
    ),
    Spacer(1, 8),
    Paragraph("What still requires manual review", heading_style),
    Paragraph(
        "Human review remains necessary for business-risk interpretation, environment-specific data, automation feasibility, "
        "and any requirement ambiguity. The agent should be treated as an accelerator and reviewer, not as the final authority.",
        body_style
    )
]
doc.build(content)
print("Reflection PDF created:", pdf_file)


Reflection PDF created: Agentic_AI_Capstone_Reflection.pdf


In [118]:
import os

print("PDF exists:", os.path.exists("Agentic_AI_Capstone_Reflection.pdf"))
print(
    "File size:",
    os.path.getsize("Agentic_AI_Capstone_Reflection.pdf"),
    "bytes"
)

PDF exists: True
File size: 2852 bytes


In [119]:
from google.colab import files

files.download("Agentic_AI_Capstone_Reflection.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 16. Review all generated files
Before submission, open/download these files and inspect them.

In [120]:
import os

print("========== FILES IN COLAB ==========\n")

for root, dirs, files in os.walk("/content"):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path)

        print(f"{path}  |  {size:,} bytes")

========== FILES IN COLAB ==========

/content/coverage_report.csv  |  968 bytes
/content/final_test_suite.csv  |  9,748 bytes
/content/Agentic_AI_Capstone_Reflection.pdf  |  2,852 bytes
/content/validation_feature_a.csv  |  371 bytes
/content/coverage_gaps.csv  |  77 bytes
/content/DESIGN_WRITEUP.md  |  1,538 bytes
/content/validation_feature_b.csv  |  479 bytes
/content/category_coverage.csv  |  69 bytes
/content/.config/.last_survey_prompt.yaml  |  37 bytes
/content/.config/.last_opt_in_prompt.yaml  |  3 bytes
/content/.config/.last_update_check.json  |  134 bytes
/content/.config/gce  |  5 bytes
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db  |  12,288 bytes
/content/.config/default_configs.db  |  12,288 bytes
/content/.config/config_sentinel  |  0 bytes
/content/.config/active_config  |  7 bytes
/content/.config/configurations/config_default  |  94 bytes
/content/.config/logs/2026.09.04/13.24.31.709670.log  |  465 bytes
/content/.config/logs/2026.0

In [121]:
import os

project_files = [
    "final_test_suite.csv",
    "validation_feature_a.csv",
    "validation_feature_b.csv",
    "category_coverage.csv",
    "coverage_report.csv",
    "coverage_gaps.csv",
    "Agentic_AI_Capstone_Reflection.pdf",
    "DESIGN_WRITEUP.md"
]

print("========== FINAL PROJECT FILE REVIEW ==========\n")

for file in project_files:

    if os.path.exists(file):
        size = os.path.getsize(file)

        print(f"✅ {file}")
        print(f"   Size: {size:,} bytes")
    else:
        print(f"❌ {file} -- NOT FOUND")

print("\n==============================================")

========== FINAL PROJECT FILE REVIEW ==========

✅ final_test_suite.csv
   Size: 9,748 bytes
✅ validation_feature_a.csv
   Size: 371 bytes
✅ validation_feature_b.csv
   Size: 479 bytes
✅ category_coverage.csv
   Size: 69 bytes
✅ coverage_report.csv
   Size: 968 bytes
✅ coverage_gaps.csv
   Size: 77 bytes
✅ Agentic_AI_Capstone_Reflection.pdf
   Size: 2,852 bytes
✅ DESIGN_WRITEUP.md
   Size: 1,538 bytes



In [122]:
import pandas as pd
import os

csv_files = [
    "final_test_suite.csv",
    "validation_feature_a.csv",
    "validation_feature_b.csv",
    "category_coverage.csv",
    "coverage_report.csv",
    "coverage_gaps.csv"
]

for file in csv_files:

    print("\n" + "=" * 70)
    print(file)
    print("=" * 70)

    if os.path.exists(file):

        df = pd.read_csv(file)

        print("Rows:", len(df))
        print("Columns:", len(df.columns))
        print("Columns:", list(df.columns))

        display(df.head(10))

    else:
        print("❌ FILE NOT FOUND")


final_test_suite.csv
Rows: 21
Columns: 11
Columns: ['Test Case ID', 'Feature', 'Acceptance Criteria', 'Test Scenario', 'Preconditions', 'Test Data', 'Test Steps', 'Expected Result', 'Category', 'Priority', 'Risk']


,Test Case ID,Feature,Acceptance Criteria,Test Scenario,Preconditions,Test Data,Test Steps,Expected Result,Category,Priority,Risk
0,TC-001,User Login,AC1,Successful login with valid credentials,"Active, registered user exists.",Email: user@example.com; Password: ValidPass123!,1. Open login page. 2. Enter email. 3. Enter p...,Dashboard is displayed. Session cookie (or tok...,Positive,P0,High
1,TC-002,User Login,AC2,Invalid password – generic error,"Active, registered user exists.",Email: user@example.com; Password: WrongPass!,1. Open login page. 2. Enter correct email. 3....,"Login page remains. Error message exactly ""Inv...",Negative,P0,High
2,TC-003,User Login,AC3,Unregistered email – same generic error,No account exists for the provided email.,Email: unknown@example.com; Password: AnyPass123!,1. Open login page. 2. Enter unregistered emai...,"Login page remains. Error message ""Invalid ema...",Negative,P0,High
3,TC-004,User Login,AC4,Empty fields validation,"Active, registered user exists.",Email: (blank); Password: (blank),1. Open login page. 2. Leave email and passwor...,Inline validation prompts to fill required fie...,Negative,P1,Medium
4,TC-005,User Login,AC5,Invalid email format,"Active, registered user exists.",Email: invalid-email; Password: ValidPass123!,1. Open login page. 2. Enter invalid email for...,"Inline message ""Enter a valid email address"" i...",Negative,P1,Medium
5,TC-006,User Login,AC6,Account lockout after consecutive failed attempts,"Active, registered user exists.",Email: user@example.com; Password: WrongPass! ...,1. Open login page. 2. Attempt login with wron...,"After fifth failure, account is locked. When c...",Negative,P0,High
6,TC-007,User Login,AC7,Case sensitivity of email and password,"Active, registered user exists.",Email: USER@EXAMPLE.COM; Password: ValidPass12...,1. Open login page. 2. Enter email with differ...,Email case variation is accepted; login succee...,Edge,P1,Medium
7,TC-008,User Login,AC8,Session persistence after successful login,User has successfully logged in.,NaN,1. Perform a successful login. 2. Refresh the ...,User remains logged in; session persists for u...,Positive,P1,Medium
8,TC-009,User Login,AC9,Login attempt with inactive account,User account is deactivated.,Email: inactive@example.com; Password: ValidPa...,1. Open login page. 2. Enter email of inactive...,"Error message ""This account is inactive. Conta...",Negative,P0,High
9,TC-001,Apply Promo Code at Checkout,AC1,Apply a valid percentage promo code (SAVE10) t...,User is on the Checkout page; cart item‑subtot...,Promo code: SAVE10,"1. Verify item‑subtotal displayed as ₹1,000.\n...",A 10% discount of ₹100 is applied; item‑subtot...,Positive,P1,Low



validation_feature_a.csv
Rows: 6
Columns: 4
Columns: ['Feature', 'Check', 'Status', 'Details']


,Feature,Check,Status,Details
0,User Login,Required Columns,PASS,All required columns present
1,User Login,Empty Required Fields,PASS,No empty required fields
2,User Login,Duplicate Test Case IDs,PASS,0 duplicate rows
3,User Login,Valid Test Categories,PASS,All categories valid
4,User Login,Valid Priorities,PASS,All priorities valid
5,User Login,Valid Risks,PASS,All risks valid



validation_feature_b.csv
Rows: 6
Columns: 4
Columns: ['Feature', 'Check', 'Status', 'Details']


...

,Category,Test Case Count
0,Negative,11
1,Positive,6
2,Edge,3
3,Boundary,1



coverage_report.csv
Rows: 21
Columns: 5
Columns: ['Feature', 'Acceptance Criteria', 'Coverage Status', 'Test Case Count', 'Test Case IDs']


,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs
0,User Login,AC1,COVERED,1,TC-001
1,User Login,AC2,COVERED,1,TC-002
2,User Login,AC3,COVERED,1,TC-003
3,User Login,AC4,COVERED,1,TC-004
4,User Login,AC5,COVERED,1,TC-005
5,User Login,AC6,COVERED,1,TC-006
6,User Login,AC7,COVERED,1,TC-007
7,User Login,AC8,COVERED,1,TC-008
8,User Login,AC9,COVERED,1,TC-009
9,Apply Promo Code at Checkout,AC1,COVERED,1,TC-001



coverage_gaps.csv
Rows: 0
Columns: 5
Columns: ['Feature', 'Acceptance Criteria', 'Coverage Status', 'Test Case Count', 'Test Case IDs']


,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs


In [ ]:
df_final = pd.read_csv("final_test_suite.csv")

print("========== FINAL TEST SUITE ==========")
print("Total Test Cases:", len(df_final))
print("Total Columns:", len(df_final.columns))

print("\nColumns:")
for col in df_final.columns:
    print("✅", col)

print("\nCategory Distribution:")
print(df_final["Category"].value_counts())

print("\nAcceptance Criteria Distribution:")
print(df_final["Acceptance Criteria"].value_counts())

display(df_final)

In [ ]:
print("========== FINAL QUALITY CHECK ==========\n")

# Empty values
empty_values = df_final.isnull().sum().sum()

print(
    "Empty values:",
    empty_values,
    "→",
    "PASS" if empty_values == 0 else "CHECK"
)

# Duplicate IDs
duplicate_ids = df_final["Test Case ID"].duplicated().sum()

print(
    "Duplicate Test Case IDs:",
    duplicate_ids,
    "→",
    "PASS" if duplicate_ids == 0 else "FAIL"
)

# Categories
allowed_categories = {
    "Positive",
    "Negative",
    "Boundary",
    "Edge"
}

actual_categories = set(
    df_final["Category"].dropna().astype(str).str.strip()
)

invalid_categories = actual_categories - allowed_categories

print(
    "Invalid Categories:",
    invalid_categories,
    "→",
    "PASS" if not invalid_categories else "FAIL"
)

In [123]:
coverage = pd.read_csv("coverage_report.csv")

print("========== COVERAGE REPORT ==========\n")

display(coverage)

if "Coverage Status" in coverage.columns:

    gaps = coverage[
        coverage["Coverage Status"].astype(str).str.lower() != "covered"
    ]

    print("\nCoverage gaps:", len(gaps))

    if len(gaps) == 0:
        print("✅ PASS — All acceptance criteria are covered.")
    else:
        print("⚠️ GAPS FOUND")
        display(gaps)

========== COVERAGE REPORT ==========



,Feature,Acceptance Criteria,Coverage Status,Test Case Count,Test Case IDs
0,User Login,AC1,COVERED,1,TC-001
1,User Login,AC2,COVERED,1,TC-002
2,User Login,AC3,COVERED,1,TC-003
3,User Login,AC4,COVERED,1,TC-004
4,User Login,AC5,COVERED,1,TC-005
5,User Login,AC6,COVERED,1,TC-006
6,User Login,AC7,COVERED,1,TC-007
7,User Login,AC8,COVERED,1,TC-008
8,User Login,AC9,COVERED,1,TC-009
9,Apply Promo Code at Checkout,AC1,COVERED,1,TC-001



Coverage gaps: 0
✅ PASS — All acceptance criteria are covered.


In [126]:
import os

pdf = "Agentic_AI_Capstone_Reflection.pdf"

if os.path.exists(pdf):

    print("✅ Reflection PDF exists")
    print("File:", pdf)
    print("Size:", os.path.getsize(pdf), "bytes")

else:

    print("❌ Reflection PDF NOT FOUND")

✅ Reflection PDF exists
File: Agentic_AI_Capstone_Reflection.pdf
Size: 2852 bytes


## 17. Download the complete outputs
Run this cell after you have reviewed the outputs.

In [125]:
import os
import zipfile
from google.colab import files

# Final project files
output_files = [
    "final_test_suite.csv",
    "validation_feature_a.csv",
    "validation_feature_b.csv",
    "category_coverage.csv",
    "coverage_report.csv",
    "coverage_gaps.csv",
    "Agentic_AI_Capstone_Reflection.pdf",
    "DESIGN_WRITEUP.md"
]

zip_name = "Agentic_AI_Capstone_Final_Outputs.zip"

# Create ZIP
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:

    for file in output_files:

        if os.path.exists(file):
            zipf.write(file)
            print("Added:", file)
        else:
            print("NOT FOUND:", file)

print("\nZIP created successfully:")
print(zip_name)

# Download ZIP
files.download(zip_name)

Added: final_test_suite.csv
Added: validation_feature_a.csv
Added: validation_feature_b.csv
Added: category_coverage.csv
Added: coverage_report.csv
Added: coverage_gaps.csv
Added: Agentic_AI_Capstone_Reflection.pdf
Added: DESIGN_WRITEUP.md

ZIP created successfully:
Agentic_AI_Capstone_Final_Outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 18. Submission checklist
Before sending the repository link:
- [ ] Both Feature A and Feature B are included.
- [ ] Every test case has an AC ID.
- [ ] Positive, Negative, Boundary and Edge cases are represented.
- [ ] Critic output identifies coverage gaps.
- [ ] Improver output addresses gaps.
- [ ] Validator result is reviewed.
- [ ] CSV, Gherkin and Excel outputs open correctly.
- [ ] Reflection PDF is reviewed.
- [ ] API key is NOT in the notebook/source files.
- [ ] README and design writeup are complete.
- [ ] GitHub repository is public or accessible to evaluator.


In [124]:
import os
import pandas as pd

print("=" * 75)
print("       AGENTIC AI CAPSTONE PROJECT")
print("              SUBMISSION CHECKLIST")
print("=" * 75)

checklist = []


def check_file(file_name, description):

    exists = os.path.exists(file_name)
    size = os.path.getsize(file_name) if exists else 0

    checklist.append({
        "Item": description,
        "File": file_name,
        "Status": "PASS" if exists and size > 0 else "FAIL",
        "Size": f"{size:,} bytes" if exists else "Missing"
    })


# =========================================================
# GENERATED OUTPUTS
# =========================================================

check_file(
    "final_test_suite.csv",
    "Final generated test suite"
)

check_file(
    "validation_feature_a.csv",
    "Feature A validation report"
)

check_file(
    "validation_feature_b.csv",
    "Feature B validation report"
)

check_file(
    "category_coverage.csv",
    "Test category coverage report"
)

check_file(
    "coverage_report.csv",
    "Acceptance criteria coverage report"
)

check_file(
    "coverage_gaps.csv",
    "Coverage gap report"
)

check_file(
    "Agentic_AI_Capstone_Reflection.pdf",
    "Reflection PDF"
)

check_file(
    "DESIGN_WRITEUP.md",
    "Design writeup"
)


# =========================================================
# DISPLAY CHECKLIST
# =========================================================

checklist_df = pd.DataFrame(checklist)

display(checklist_df)


# =========================================================
# QUALITY CHECKS
# =========================================================

print("\n" + "=" * 75)
print("              QUALITY CHECKS")
print("=" * 75)

if os.path.exists("final_test_suite.csv"):

    final_df = pd.read_csv("final_test_suite.csv")

    print("\nTotal test cases:", len(final_df))

    # -----------------------------------------------------
    # Required columns
    # -----------------------------------------------------

    required_columns = [
        "Test Case ID",
        "Feature",
        "Acceptance Criteria",
        "Category",
        "Test Scenario",
        "Preconditions",
        "Test Steps",
        "Test Data",
        "Expected Result"
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in final_df.columns
    ]

    if len(missing_columns) == 0:
        print("Required columns: PASS")
    else:
        print("Required columns: FAIL")
        print("Missing:", missing_columns)

    # -----------------------------------------------------
    # Duplicate IDs
    # -----------------------------------------------------

    if "Test Case ID" in final_df.columns:

        duplicates = final_df["Test Case ID"].duplicated().sum()

        if duplicates == 0:
            print("Duplicate Test Case IDs: PASS")
        else:
            print(
                f"Duplicate Test Case IDs: FAIL ({duplicates})"
            )

    # -----------------------------------------------------
    # Categories
    # -----------------------------------------------------

    if "Category" in final_df.columns:

        allowed_categories = {
            "Positive",
            "Negative",
            "Boundary",
            "Edge"
        }

        actual_categories = set(
            final_df["Category"]
            .dropna()
            .astype(str)
            .str.strip()
        )

        invalid_categories = (
            actual_categories - allowed_categories
        )

        if len(invalid_categories) == 0:
            print("Test categories: PASS")
        else:
            print("Test categories: FAIL")
            print("Invalid:", invalid_categories)

        print("\nCategory distribution:")
        print(
            final_df["Category"]
            .value_counts()
        )

else:

    print("final_test_suite.csv not found")


# =========================================================
# COVERAGE CHECK
# =========================================================

print("\n" + "=" * 75)
print("              COVERAGE CHECK")
print("=" * 75)

if os.path.exists("coverage_gaps.csv"):

    gaps_df = pd.read_csv("coverage_gaps.csv")

    if len(gaps_df) == 0:

        print("Acceptance criteria coverage: PASS")
        print("No coverage gaps found.")

    else:

        print(
            f"Coverage gaps found: {len(gaps_df)}"
        )

        display(gaps_df)

else:

    print("coverage_gaps.csv not found")


# =========================================================
# FINAL RESULT
# =========================================================

print("\n" + "=" * 75)
print("                 FINAL RESULT")
print("=" * 75)

failed = checklist_df[
    checklist_df["Status"] == "FAIL"
]

if len(failed) == 0:

    print("ALL REQUIRED OUTPUT FILES ARE PRESENT")
    print("PROJECT OUTPUT CHECK PASSED")
    print("\nYOUR OUTPUT PACKAGE IS READY FOR REVIEW")

else:

    print("SUBMISSION CHECK FAILED")
    print("\nMissing files:")

    for file in failed["File"]:
        print(file)

print("=" * 75)

       AGENTIC AI CAPSTONE PROJECT
              SUBMISSION CHECKLIST


,Item,File,Status,Size
0,Final generated test suite,final_test_suite.csv,PASS,"9,748 bytes"
1,Feature A validation report,validation_feature_a.csv,PASS,371 bytes
2,Feature B validation report,validation_feature_b.csv,PASS,479 bytes
3,Test category coverage report,category_coverage.csv,PASS,69 bytes
4,Acceptance criteria coverage report,coverage_report.csv,PASS,968 bytes
5,Coverage gap report,coverage_gaps.csv,PASS,77 bytes
6,Reflection PDF,Agentic_AI_Capstone_Reflection.pdf,PASS,"2,852 bytes"
7,Design writeup,DESIGN_WRITEUP.md,PASS,"1,538 bytes"



              QUALITY CHECKS

Total test cases: 21
Required columns: PASS
Duplicate Test Case IDs: FAIL (9)
Test categories: PASS

Category distribution:
Category
Negative    11
Positive     6
Edge         3
Boundary     1
Name: count, dtype: int64

              COVERAGE CHECK
Acceptance criteria coverage: PASS
No coverage gaps found.

                 FINAL RESULT
ALL REQUIRED OUTPUT FILES ARE PRESENT
PROJECT OUTPUT CHECK PASSED

YOUR OUTPUT PACKAGE IS READY FOR REVIEW
